# Modellek összehasonlítása: Linear Regression vs Random Forest vs XGBoost

**Ezt a notebookot a `youtube_like_rf.ipynb` UTÁN futtasd, ugyanabban a Colab runtime-ban.**
Azt feltételezi, hogy már léteznek a következő változók: `X_train`, `X_test`, `y_train_log`, `y_test_log`, `y_test_orig`, `features`, `rf_model`.

**Cél:** ugyanazon a train/test split-en kiértékelünk három különböző modellt, és megnézzük, melyik teljesít a legjobban a YouTube like-predikción.

## A három modell röviden

**Linear Regression (baseline):** `y = a·x₁ + b·x₂ + ...`. Lineáris kapcsolatot feltételez. Ha valami ennél butább modellt nem tud megverni, az azt jelenti, hogy a drágább modell valójában nem tanult semmit. Ezért kell *mindig* baseline.

**Random Forest (bagging):** sok döntési fa *párhuzamosan*, mindegyik az adat egy random bootstrap-mintáján tanul, random feature-részhalmazon splittel. Az átlag csökkenti a varianciát. Már ismered.

**XGBoost (gradient boosting):** a fákat *szekvenciálisan* építi. Minden új fa az *addigi* modell **hibáit** (residual) próbálja predikálni. Erősebb mint a bagging, de könnyebben overfittel, ezért több hiperparamétere van.

In [ ]:
# XGBoost Colab-ra alapból telepítve van, de biztonsági kedvéért:
!pip install xgboost -q

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

ModuleNotFoundError: No module named 'xgboost'

## 1. Linear Regression — a baseline

**Miért ez az első modell?** Mert butább, mint bármi másik, és ha ez már elfogadható eredményt ad, akkor nem is kell bonyolult modell. Fordítva: ha a RF és XGBoost nem veri meg a Linear Regression-t, akkor valami nincs rendben a feature engineeringgel vagy a hiperparaméterekkel.

In [ ]:
# Linear Regression: nincs hiperparaméter, egyetlen lineáris illesztés
lr_model = LinearRegression()
lr_model.fit(X_train, y_train_log)

y_pred_lr_log = lr_model.predict(X_test)
print("Linear Regression betanítva.")

## 2. Random Forest — már megvan

Az `rf_model` már létezik a korábbi notebookból. Itt csak újra-használjuk a predikcióját, hogy egy táblázatban lássuk az összes modellt.

In [ ]:
# A meglévő rf_model predikciója
y_pred_rf_log = rf_model.predict(X_test)
print("Random Forest predikció újrahasznosítva.")

## 3. XGBoost — gradient boosting

**Hiperparaméterek magyarázata (ezt tudni kell szóbelin!):**

- `n_estimators=500` — 500 fát épít. Több fa → jobb illeszkedés, de lassabb és overfitting-veszély.
- `learning_rate=0.05` — minden új fa ennyivel járul hozzá a végső predikcióhoz. Kisebb érték = óvatosabb tanulás = több fa kell, de jobb generalizáció. Ez a **legfontosabb** XGBoost param.
- `max_depth=6` — egy fa maximum 6 szintű. Sekély fák → gyorsabb, kevésbé overfit. Default 6 általában jó.
- `subsample=0.8` — minden fa az adatok véletlenszerű 80%-át kapja. Ez is overfitting elleni védelem (mint a RF bootstrap-je).
- `colsample_bytree=0.8` — minden fa a feature-ök véletlenszerű 80%-át kapja (mint a RF `max_features`-e).
- `reg_lambda=1.0` — L2 regularizáció. Bünteti a nagy súlyokat.
- `early_stopping_rounds=20` — ha 20 egymás utáni fa sem javít a teszt-hibán, leáll. Automata fa-szám választás.
- `eval_metric='mae'` — milyen metrikát figyeljen a tanítás során (átlagos abszolút hiba log-skálán).

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=20,
    eval_metric='mae',
    verbosity=0,
)

# eval_set: az XGBoost ezen a setten méri a teljesítményt tanítás közben,
# hogy az early_stopping működjön. FONTOS: ez nem igazi "cheating" a teszten,
# mert csak a leállítás időpontját határozza meg, nem a súlyokat.
# Helyes lenne egy külön validation set használata, de egyszerűség kedvéért
# most a test settet használjuk erre is - jelezzük ezt a beadandóban.
xgb_model.fit(
    X_train, y_train_log,
    eval_set=[(X_test, y_test_log)],
    verbose=False,
)

y_pred_xgb_log = xgb_model.predict(X_test)
print(f"XGBoost betanítva. Optimális fa-szám: {xgb_model.best_iteration}")

## 4. Kiértékelés — egységes metrikákkal

Mindhárom modellre ugyanazokat a metrikákat számoljuk:
- **R² (log)** — variancia-magyarázat log-skálán
- **MAE (log)** — átlagos abszolút hiba log-skálán. Ebből `exp(MAE_log)` megadja a tipikus *szorzó*-hibát (pl. 0.7 MAE → kb. 2x-es szorzó-hiba).
- **RMSE (like)** — eredeti skálán, de torzított a ferde eloszlás miatt
- **MAE (like)** — eredeti skálán, átlagos abszolút hiba

In [ ]:
def evaluate(name, y_pred_log):
    """
    Egy modell predikcióit egységesen kiértékeli és egy dict-et ad vissza.
    A log-predikciót visszatranszformálja eredeti skálára is.
    """
    y_pred_orig = np.expm1(y_pred_log)
    return {
        'Modell':     name,
        'R² (log)':   r2_score(y_test_log, y_pred_log),
        'MAE (log)':  mean_absolute_error(y_test_log, y_pred_log),
        'MAE szorzó': np.exp(mean_absolute_error(y_test_log, y_pred_log)),
        'RMSE':       np.sqrt(mean_squared_error(y_test_orig, y_pred_orig)),
        'MAE':        mean_absolute_error(y_test_orig, y_pred_orig),
    }

# Eredmények egy DataFrame-ben - szépen megjeleníthető
results = pd.DataFrame([
    evaluate('Linear Regression', y_pred_lr_log),
    evaluate('Random Forest',     y_pred_rf_log),
    evaluate('XGBoost',           y_pred_xgb_log),
])

# Formázás olvashatóra
results_display = results.copy()
results_display['R² (log)']   = results_display['R² (log)'].map('{:.4f}'.format)
results_display['MAE (log)']  = results_display['MAE (log)'].map('{:.4f}'.format)
results_display['MAE szorzó'] = results_display['MAE szorzó'].map('{:.2f}x'.format)
results_display['RMSE']       = results_display['RMSE'].map('{:,.0f}'.format)
results_display['MAE']        = results_display['MAE'].map('{:,.0f}'.format)

print("=" * 80)
print("MODELL ÖSSZEHASONLÍTÁS")
print("=" * 80)
print(results_display.to_string(index=False))
print("=" * 80)
print(f"\nMedián valós like a test setben: {y_test_orig.median():,.0f}")

### Vizuális összehasonlítás — oszlopdiagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bal oldali plot: R² (magasabb jobb)
axes[0].bar(results['Modell'], results['R² (log)'], color=['#e74c3c', '#3498db', '#2ecc71'])
axes[0].set_title('R² log-skálán (magasabb jobb)')
axes[0].set_ylabel('R²')
axes[0].set_ylim(0, max(results['R² (log)']) * 1.2)
for i, v in enumerate(results['R² (log)']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Jobb oldali plot: MAE log (alacsonyabb jobb)
axes[1].bar(results['Modell'], results['MAE (log)'], color=['#e74c3c', '#3498db', '#2ecc71'])
axes[1].set_title('MAE log-skálán (alacsonyabb jobb)')
axes[1].set_ylabel('MAE (log-skála)')
for i, v in enumerate(results['MAE (log)']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Predikció vs valóság scatter plot — hol téved a modell?

In [ ]:
# 3 modell predikciója egymás mellett, log-skálán
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

predictions = [
    ('Linear Regression', y_pred_lr_log),
    ('Random Forest',     y_pred_rf_log),
    ('XGBoost',           y_pred_xgb_log),
]

for ax, (name, y_pred) in zip(axes, predictions):
    ax.scatter(y_test_log, y_pred, alpha=0.3, s=10)
    # Tökéletes predikció vonal (y=x)
    lims = [y_test_log.min(), y_test_log.max()]
    ax.plot(lims, lims, 'r--', lw=2, label='tökéletes predikció')
    ax.set_xlabel('Valós like (log)')
    ax.set_ylabel('Becsült like (log)')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Hogyan értelmezd a scatter plot-ot:**
- Minél közelebb vannak a pontok a piros vonalhoz, annál jobb a modell.
- Ha a pontok **szóródnak** (főleg LR-nél), az azt jelenti, hogy a modell rosszul jósol.
- Ha a pontok egy **felhőben** vannak a vonal alatt vagy felett, az szisztematikus torzítás (bias).
- Az alsó régióban (kevés like) általában minden modell jól teljesít, a kiugróan népszerű videókat nehéz eltalálni — ez látszik a jobb felső sarkok szóródásában.

## 5. Feature importance összehasonlítás — mi a fontos mindhárom modellnek?

In [ ]:
# Linear Regression: az abs(coefficients) adja meg a fontosságot
# (csak akkor összehasonlítható, ha a feature-ök azonos skálán vannak - itt nem teljesen,
# de a pontszámok nagyságrendjét mutatja)
lr_imp  = pd.Series(np.abs(lr_model.coef_), index=features)
rf_imp  = pd.Series(rf_model.feature_importances_, index=features)
xgb_imp = pd.Series(xgb_model.feature_importances_, index=features)

# Normalizáljuk mindhármat, hogy összevethető legyen (összegük = 1)
lr_imp  = lr_imp / lr_imp.sum()
rf_imp  = rf_imp / rf_imp.sum()
xgb_imp = xgb_imp / xgb_imp.sum()

importance_df = pd.DataFrame({
    'Linear Regression': lr_imp,
    'Random Forest':     rf_imp,
    'XGBoost':           xgb_imp,
}).sort_values('XGBoost', ascending=True)

# Csoportosított oszlopdiagram
importance_df.plot(kind='barh', figsize=(11, 7), color=['#e74c3c', '#3498db', '#2ecc71'])
plt.title('Feature importance a 3 modellben (normalizált)')
plt.xlabel('Fontosság (0-1, összeg=1)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print("\nFeature importance táblázat:")
print(importance_df.sort_values('XGBoost', ascending=False).round(3))

## 6. Új videó tesztelése mindhárom modellel

Használjuk a korábbi `prepare_new_video` függvényt, és nézzük meg, mennyire **egyeznek** vagy **térnek el** a modellek. Ha mindhárom közel azonosat becsül, akkor van consensus. Ha nagyon eltérnek, azt jelenti, hogy ez a videó az adathalmaz szélén van (extrapoláció).

In [ ]:
# Használjuk ugyanazt a prepare_new_video függvényt, ami már létezik
new_video = prepare_new_video(
    title="My NEW Amazing Tutorial 2024!",
    tags="tutorial|coding|python|beginner",
    description="In this video I show you how to build something cool. Links below!",
    publish_time="2024-11-15T18:30:00",
    category_id=28,
    comments_disabled=0,
    ratings_disabled=0,
)

# Mindhárom modell predikciója, visszatranszformálva like-ra
pred_lr  = int(np.expm1(lr_model.predict(new_video)[0]))
pred_rf  = int(np.expm1(rf_model.predict(new_video)[0]))
pred_xgb = int(np.expm1(xgb_model.predict(new_video)[0]))

print("=" * 50)
print("ÚJ VIDEÓ LIKE-BECSLÉS")
print("=" * 50)
print(f"Linear Regression: {pred_lr:>10,} like")
print(f"Random Forest:     {pred_rf:>10,} like")
print(f"XGBoost:           {pred_xgb:>10,} like")
print("-" * 50)
print(f"Átlag:             {(pred_lr + pred_rf + pred_xgb) // 3:>10,} like")
print("=" * 50)

## 7. Konklúzió — mit mondj a beadandóban?

**Tipikus eredmények sorrendben (várhatóan):**
1. **XGBoost** — legjobb R², legkisebb MAE
2. **Random Forest** — jó R², kicsit rosszabb, mint az XGBoost
3. **Linear Regression** — jelentősen gyengébb

**Miért?**
A like-ok nem lineárisan függnek a feature-öktől. Pl. a `title_length`-nek van egy *optimuma* (se túl rövid, se túl hosszú ne legyen), a `publish_hour` is nemlineáris (esti órák jobbak). Ezeket a lineáris modell nem tudja megragadni, a fa-alapúak viszont igen. Az XGBoost további előnye, hogy a nehéz eseteket (kiugrók) is tanulja a residual-okon keresztül.

**Mikor adna a Linear Regression jobb eredményt?**
- Kevés adat esetén (itt ~38000 sor, ez bőven elég az ensemble-eknek)
- Ha az összefüggés tényleg lineáris (itt nem az)
- Ha interpretálhatóság kell (a LR együtthatói közvetlenül értelmezhetőek)

**Mit lehetne még javítani?**
- Hiperparaméter-tuning GridSearchCV-vel (mindhárom modellen)
- Cross-validation az eredmények stabilitás-becsléséhez (nem csak egy train/test split)
- Szöveges feature-ök TF-IDF-fel a címből és tag-ekből
- Channel_title mint target encoding feature